# previous approach


In [2]:
!pip install PyPDF2

  Using cached pypdf2-3.0.1-py3-none-any.whl.metadata (6.8 kB)
Using cached pypdf2-3.0.1-py3-none-any.whl (232 kB)


In [10]:
import re
from PyPDF2 import PdfReader

def extract_text_from_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

def extract_clauses(text):
    # Split by form sections and attachments
    attachments = re.split(r'ATTACHMENT [A-Z]|RFP \d{2}-\d{3}', text)
    
    # Process each attachment/section
    clauses = []
    for section in attachments:
        if len(section.strip()) > 0:
            # Split by form fields and numbered items
            items = re.split(r'\n\s*\d+\.\s+|\n\s*•\s+|\n{2,}', section)
            # Clean and filter items
            items = [item.strip() for item in items if len(item.strip()) > 30]
            clauses.extend(items)
            
    return clauses


In [ ]:
text = extract_text_from_pdf("data/ELIGIBLE_RFP_1.pdf")
print(text)

RFP 25-008  Temporary Staffing Services Page 1 of 33                REQUEST FOR PROPOSAL   TEMPORARY STAFFING SERVICES   RFP 25-008    BIDS DUE THURSDAY 27, 2025  AT 2:00 PM     MHMR OF TARRANT COUNTY (“MHMR”) PROCUREMENT SERVICES   3840 HULEN STREET FORT WORTH, TEXAS 76107    

RFP 25-008  Temporary Staffing Services Page 2 of 33  PRE-PROPOSAL CONFERENCE DATE AND TIME    All vendors must attend the scheduled Pre-Proposal Conference in order to get a clear understanding of the requirements of this RFP:   DATE:  February 12, 2025    TIME:  10:00 A. M.     LOCATION:  3840 Hulen St. Fort Worth, TX 76107  Send RSVP to Procurement Services Department fax at (817) 810-3100 or email mhmr.purchasing@mhmrtc.org for schedule.  RSVP:  
 
Company Name: _________________ _________________________________________ ________  
 
Contact Name____________________ _________________________________________ _______  
 
Planning to attend Pre -Bid Meeting:  _______ ___YES      _______ ___NO  
 
If yes, numbe

In [12]:
clauses = extract_clauses(text)

for i, clause in enumerate(clauses[:5], 1):
    print(f"Clause {i}: {clause}\n")

Clause 1: Temporary Staffing Services Page 1 of 33                REQUEST FOR PROPOSAL   TEMPORARY STAFFING SERVICES

Clause 2: BIDS DUE THURSDAY 27, 2025  AT 2:00 PM     MHMR OF TARRANT COUNTY (“MHMR”) PROCUREMENT SERVICES   3840 HULEN STREET FORT WORTH, TEXAS 76107

Clause 3: Temporary Staffing Services Page 2 of 33  PRE-PROPOSAL CONFERENCE DATE AND TIME    All vendors must attend the scheduled Pre-Proposal Conference in order to get a clear understanding of the requirements of this RFP:   DATE:  February 12, 2025    TIME:  10:00 A. M.     LOCATION:  3840 Hulen St. Fort Worth, TX 76107  Send RSVP to Procurement Services Department fax at (817) 810-3100 or email mhmr.purchasing@mhmrtc.org for schedule.  RSVP:  
 
Company Name: _________________ _________________________________________ ________  
 
Contact Name____________________ _________________________________________ _______  
 
Planning to attend Pre -Bid Meeting:  _______ ___YES      _______ ___NO  
 
If yes, number of represen

In [13]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=["ATTACHMENT", "\n\n", "\n", ".", ";"],
    chunk_size=300,  # Reduced chunk size for forms
    chunk_overlap=20
)
form_clauses = text_splitter.split_text(text)

# Print results
for i, clause in enumerate(form_clauses[:10], 1):
    print(f"Section {i}:\n{clause}\n")

Section 1:
RFP 25-008  Temporary Staffing Services Page 1 of 33                REQUEST FOR PROPOSAL   TEMPORARY STAFFING SERVICES   RFP 25-008    BIDS DUE THURSDAY 27, 2025  AT 2:00 PM     MHMR OF TARRANT COUNTY (“MHMR”) PROCUREMENT SERVICES   3840 HULEN STREET FORT WORTH, TEXAS 76107

Section 2:
RFP 25-008  Temporary Staffing Services Page 2 of 33  PRE-PROPOSAL CONFERENCE DATE AND TIME    All vendors must attend the scheduled Pre-Proposal Conference in order to get a clear understanding of the requirements of this RFP:   DATE:  February 12, 2025    TIME:  10:00 A. M

Section 3:
. M.     LOCATION:  3840 Hulen St. Fort Worth, TX 76107  Send RSVP to Procurement Services Department fax at (817) 810-3100 or email mhmr.purchasing@mhmrtc.org for schedule.  RSVP:

Section 4:
Company Name: _________________ _________________________________________ ________  
 
Contact Name____________________ _________________________________________ _______  
 
Planning to attend Pre -Bid Meeting:  _______ _

# New approach


In [14]:
from PyPDF2 import PdfReader

def extract_text_from_pdf(pdf_file):
    reader = PdfReader(pdf_file)
    text = "\n".join([page.extract_text() for page in reader.pages])
    return text


In [16]:
def extract_text_from_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text
text = extract_text_from_pdf("data/ELIGIBLE_RFP_1.pdf")
print(text)

RFP 25-008  Temporary Staffing Services Page 1 of 33                REQUEST FOR PROPOSAL   TEMPORARY STAFFING SERVICES   RFP 25-008    BIDS DUE THURSDAY 27, 2025  AT 2:00 PM     MHMR OF TARRANT COUNTY (“MHMR”) PROCUREMENT SERVICES   3840 HULEN STREET FORT WORTH, TEXAS 76107    

RFP 25-008  Temporary Staffing Services Page 2 of 33  PRE-PROPOSAL CONFERENCE DATE AND TIME    All vendors must attend the scheduled Pre-Proposal Conference in order to get a clear understanding of the requirements of this RFP:   DATE:  February 12, 2025    TIME:  10:00 A. M.     LOCATION:  3840 Hulen St. Fort Worth, TX 76107  Send RSVP to Procurement Services Department fax at (817) 810-3100 or email mhmr.purchasing@mhmrtc.org for schedule.  RSVP:  
 
Company Name: _________________ _________________________________________ ________  
 
Contact Name____________________ _________________________________________ _______  
 
Planning to attend Pre -Bid Meeting:  _______ ___YES      _______ ___NO  
 
If yes, numbe

In [18]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ".", ";"],
    chunk_size=3000,
    chunk_overlap=300
)

chunks = splitter.split_text(text)


In [25]:
import requests

def extract_clauses_llm(text_chunk, api_key):
    prompt = f"""
You are a legal assistant. Extract all the individual legal or contractual clauses from this text.

Return only a list of clauses.

Text:
\"\"\"{text_chunk}\"\"\"
"""
    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}"},
        json={
            "model": "llama3-70b-8192",  # Or your preferred model
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.3
        }
    )
    print("response:", response.json())
    return response.json()["choices"][0]["message"]["content"]


In [23]:
chunks

['RFP 25-008  Temporary Staffing Services Page 1 of 33                REQUEST FOR PROPOSAL   TEMPORARY STAFFING SERVICES   RFP 25-008    BIDS DUE THURSDAY 27, 2025  AT 2:00 PM     MHMR OF TARRANT COUNTY (“MHMR”) PROCUREMENT SERVICES   3840 HULEN STREET FORT WORTH, TEXAS 76107',
 'RFP 25-008  Temporary Staffing Services Page 2 of 33  PRE-PROPOSAL CONFERENCE DATE AND TIME    All vendors must attend the scheduled Pre-Proposal Conference in order to get a clear understanding of the requirements of this RFP:   DATE:  February 12, 2025    TIME:  10:00 A. M.     LOCATION:  3840 Hulen St. Fort Worth, TX 76107  Send RSVP to Procurement Services Department fax at (817) 810-3100 or email mhmr.purchasing@mhmrtc.org for schedule.  RSVP:  \n \nCompany Name: _________________ _________________________________________ ________  \n \nContact Name____________________ _________________________________________ _______  \n \nPlanning to attend Pre -Bid Meeting:  _______ ___YES      _______ ___NO  \n \nIf y

In [27]:
for chunk in chunks:
    extract_clauses_llm(text_chunk=chunk, api_key="REMOVED_GROQ_API_KEY")

response: {'id': 'chatcmpl-bb2218ad-45ac-4e6f-ac9c-fc14ec88e23e', 'object': 'chat.completion', 'created': 1743886870, 'model': 'llama3-70b-8192', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Here is the list of individual legal or contractual clauses extracted from the text:\n\n* None.'}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'queue_time': 0.059606719999999995, 'prompt_tokens': 130, 'prompt_time': 0.003892424, 'completion_tokens': 19, 'completion_time': 0.054285714, 'total_tokens': 149, 'total_time': 0.058178138}, 'usage_breakdown': {'models': None}, 'system_fingerprint': 'fp_dd4ae1c591', 'x_groq': {'id': 'req_01jr3w3t5tfp18gscj7gta6z5r'}}
response: {'id': 'chatcmpl-03d916b4-ef6d-4976-9845-3b0c27e445e7', 'object': 'chat.completion', 'created': 1743886871, 'model': 'llama3-70b-8192', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Here is the list of individual legal or contractual clauses extracted from the text:\n\n1. Att

KeyError: 'choices'

In [26]:
extract_clauses_llm(text_chunk=chunks, api_key="REMOVED_GROQ_API_KEY")

response: {'error': {'message': 'Request too large for model `llama3-70b-8192` in organization `org_01jqtqxcnvefza4dmrenc9g7f6` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 14758, please reduce your message size and try again. Visit https://console.groq.com/docs/rate-limits for more information.', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


KeyError: 'choices'

# Advance approach

In [31]:
!pip install PyMuPDF
!pip install sentence-transformers

In [33]:
pip uninstall scipy scikit-learn sentence-transformers transformers -y

Found existing installation: scipy 1.15.0
Uninstalling scipy-1.15.0:
  Successfully uninstalled scipy-1.15.0
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Found existing installation: sentence-transformers 3.3.1
Uninstalling sentence-transformers-3.3.1:
  Successfully uninstalled sentence-transformers-3.3.1
Found existing installation: transformers 4.48.0
Uninstalling transformers-4.48.0:
  Successfully uninstalled transformers-4.48.0
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.
You can safely remove it manually.
You can safely remove it manually.


In [35]:
!pip install scipy==1.11.3
!pip install scikit-learn==1.3.2
!pip install transformers==4.35.2
!pip install sentence-transformers==2.2.2

   ---------------------------------------- 0.0/43.7 MB ? eta -:--:--
    --------------------------------------- 0.8/43.7 MB 4.2 MB/s eta 0:00:11
   - -------------------------------------- 1.8/43.7 MB 4.6 MB/s eta 0:00:10
   -- ------------------------------------- 2.9/43.7 MB 4.7 MB/s eta 0:00:09
   --- ------------------------------------ 3.7/43.7 MB 4.6 MB/s eta 0:00:09
   ---- ----------------------------------- 4.7/43.7 MB 4.5 MB/s eta 0:00:09
   ----- ---------------------------------- 5.8/43.7 MB 4.6 MB/s eta 0:00:09
   ------ --------------------------------- 6.8/43.7 MB 4.6 MB/s eta 0:00:09
   ------ --------------------------------- 7.3/43.7 MB 4.6 MB/s eta 0:00:08
   ------- -------------------------------- 8.4/43.7 MB 4.4 MB/s eta 0:00:09
   -------- ------------------------------- 9.4/43.7 MB 4.4 MB/s eta 0:00:08
   --------- ------------------------------ 10.2/43.7 MB 4.5 MB/s eta 0:00:08
   ---------- ----------------------------- 11.3/43.7 MB 4.4 MB/s eta 0:00:08
   -

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unstructured-inference 0.8.10 requires transformers>=4.25.1, which is not installed.


   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.1 MB 4.2 MB/s eta 0:00:02
   -------- ------------------------------- 1.8/9.1 MB 4.4 MB/s eta 0:00:02
   ----------- ---------------------------- 2.6/9.1 MB 4.4 MB/s eta 0:00:02
   ---------------- ----------------------- 3.7/9.1 MB 4.5 MB/s eta 0:00:02
   -------------------- ------------------- 4.7/9.1 MB 4.5 MB/s eta 0:00:01
   ------------------------ --------------- 5.5/9.1 MB 4.4 MB/s eta 0:00:01
   --------------------------- ------------ 6.3/9.1 MB 4.2 MB/s eta 0:00:01
   -------------------------------- ------- 7.3/9.1 MB 4.3 MB/s eta 0:00:01
   ----------------------------------- ---- 8.1/9.1 MB 4.3 MB/s eta 0:00:01
   ---------------------------------------- 9.1/9.1 MB 4.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/7.9 MB ? eta -:--:--
   --- ------------------------------------ 0.8/7.9 MB 4.2 MB/s eta 0:00:02
   ------- ---------------

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-huggingface 0.1.2 requires sentence-transformers>=2.6.0, which is not installed.
langchain-huggingface 0.1.2 requires tokenizers>=0.19.1, but you have tokenizers 0.15.2 which is incompatible.
langchain-huggingface 0.1.2 requires transformers>=4.39.0, but you have transformers 4.35.2 which is incompatible.


  Using cached sentence-transformers-2.2.2.tar.gz (85 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/992.0 kB ? eta -:--:--
   ------------------------------- -------- 786.4/992.0 kB 3.4 MB/s eta 0:00:01
   ---------------------------------------- 992.0/992.0 kB 3.3 MB/s eta 0:00:00
  Created wheel for sentence-transformers: filename=sentence_transformers-2.2.2-py3-none-any.whl size=126077 sha256=0f073b87666a6479c982e659e9f8e5f102349fb486db74d3b78d40b18ce16ab0
  Stored in directory: c:\users\anish\appdata\local\pip\cache\wheels\d9\3b\21\aa025e9c81a6cda4b8358756a756677b0969b4bc69be6dd5da
Successfully built sentence-transformers


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-huggingface 0.1.2 requires sentence-transformers>=2.6.0, but you have sentence-transformers 2.2.2 which is incompatible.
langchain-huggingface 0.1.2 requires tokenizers>=0.19.1, but you have tokenizers 0.15.2 which is incompatible.
langchain-huggingface 0.1.2 requires transformers>=4.39.0, but you have transformers 4.35.2 which is incompatible.


In [38]:
pip install -U sentence-transformers

  Using cached sentence_transformers-4.0.2-py3-none-any.whl.metadata (13 kB)
  Using cached huggingface_hub-0.30.1-py3-none-any.whl.metadata (13 kB)
  Using cached tokenizers-0.21.1-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
Using cached sentence_transformers-4.0.2-py3-none-any.whl (340 kB)
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   --- ------------------------------------ 0.8/10.4 MB 4.2 MB/s eta 0:00:03
   ------- -------------------------------- 1.8/10.4 MB 4.4 MB/s eta 0:00:02
   ---------- ----------------------------- 2.6/10.4 MB 4.4 MB/s eta 0:00:02
   ------------- -------------------------- 3.4/10.4 MB 4.3 MB/s eta 0:00:02
   ----------------- ---------------------- 4.5/10.4 MB 4.3 MB/s eta 0:00:02
   -------------------- ------------------- 5.2/10.4 MB 4.3 MB/s eta 0:00:02
   ------------------------ --------------- 6.3/10.4 MB 4.3 MB/s eta 0:00:01
   --------------------------- ------------ 7.1/10.4 MB 4.3 MB/s eta 0:00:01
   --------------------

In [1]:
import logging
logging.basicConfig(level=logging.INFO)

try:
    import fitz  # PyMuPDF
    # from sentence_transformers import SentenceTransformer, util
    from sentence_transformers import SentenceTransformer

except ImportError as e:
    logging.error(f"Import error: {e}")
    logging.info("Please make sure all required packages are installed correctly")
    raise

def extract_text(pdf_path):
    try:
        doc = fitz.open(pdf_path)
        text = "\n".join(page.get_text() for page in doc)
        doc.close()
        return text
    except Exception as e:
        logging.error(f"Error extracting text from PDF: {str(e)}")
        return None

def semantic_chunks(text, model_name="sentence-transformers/all-MiniLM-L6-v2", threshold=0.7):
    try:
        model = SentenceTransformer(model_name)
        sentences = text.split("\n")
        if not sentences:
            return []
            
        embeddings = model.encode(sentences, convert_to_tensor=True)
        
        chunks, current_chunk = [], []
        for i in range(len(sentences)-1):
            sim = model.similarity(embeddings[i], embeddings[i+1]).item()
            current_chunk.append(sentences[i])
            if sim < threshold:
                chunks.append("\n".join(current_chunk))
                current_chunk = []
        current_chunk.append(sentences[-1])
        chunks.append("\n".join(current_chunk))
        return chunks
    except Exception as e:
        logging.error(f"Error in semantic chunking: {str(e)}")
        return []

c:\Users\Anish\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import requests

def classify_clause(text, api_key):
    prompt = f"""
Classify the following text into one of the following: "Clause", "Not a Clause", "Ambiguous".

Text:
\"\"\"{text}\"\"\"
Answer:"""

    res = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}"},
        json={
            "model": "llama3-70b-8192",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0
        }
    )
    return res.json()['choices'][0]['message']['content']


In [3]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# Load Legal NER pipeline
def setup_ner_pipeline():
    model = AutoModelForTokenClassification.from_pretrained("nlpaueb/legal-bert-base-uncased")
    tokenizer = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
    return pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

# Run on a chunk
def extract_ner_entities(text, ner_pipeline):
    return ner_pipeline(text)


In [4]:
def detect_risks(text):
    risk_keywords = ["unilateral", "termination without cause", "indemnify", "no liability", "sole discretion"]
    risky = any(keyword.lower() in text.lower() for keyword in risk_keywords)
    return "⚠️ Risky Clause" if risky else "✔️ OK"


In [6]:
!pip install streamlit

  Using cached streamlit-1.44.1-py3-none-any.whl.metadata (8.9 kB)
  Using cached altair-5.5.0-py3-none-any.whl.metadata (11 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
  Using cached GitPython-3.1.44-py3-none-any.whl.metadata (13 kB)
  Using cached pydeck-0.9.1-py2.py3-none-any.whl.metadata (4.1 kB)
  Using cached narwhals-1.33.0-py3-none-any.whl.metadata (9.2 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached smmap-5.0.2-py3-none-any.whl.metadata (4.3 kB)
Using cached streamlit-1.44.1-py3-none-any.whl (9.8 MB)
Using cached altair-5.5.0-py3-none-any.whl (731 kB)
Using cached GitPython-3.1.44-py3-none-any.whl (207 kB)
Using cached pydeck-0.9.1-py2.py3-none-any.whl (6.9 MB)
Using cached toml-0.10.2-py2.py3-none-any.whl (16 kB)
Using cached watchdog-6.0.0-py3-none-win_amd64.whl (79 kB)
Using cached gitdb-4.0.12-py3-none-any.whl (62 kB)
Using cached narwhals-1.33.

In [8]:
pip install utils


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for utils: filename=utils-1.0.2-py2.py3-none-any.whl size=14010 sha256=41158d905d9ba63e2062d1443aefb92dc87e7e7acbac316e2be2c6e69572eb3e
  Stored in directory: c:\users\anish\appdata\local\pip\cache\wheels\b6\a1\81\1036477786ae0e17b522f6f5a838f9bc4288d1016fc5d0e1ec
Successfully built utils


In [9]:
# streamlit_app.py
import streamlit as st
from utils import extract_text, semantic_chunks, classify_clause, detect_risks, extract_ner_entities, setup_ner_pipeline

st.title("⚖️ Legal Contract Clause Analyzer")

api_key = st.text_input("Groq API Key", type="password")
uploaded_file = st.file_uploader("Upload Contract PDF")

if uploaded_file and api_key:
    with st.spinner("Analyzing..."):
        text = extract_text(uploaded_file)
        chunks = semantic_chunks(text)
        ner = setup_ner_pipeline()

        for chunk in chunks:
            clause_type = classify_clause(chunk, api_key)
            if "Clause" in clause_type:
                st.markdown(f"### 📜 Clause")
                st.write(chunk)
                st.markdown(f"**Classification:** `{clause_type.strip()}`")
                st.markdown(f"**Risk Check:** `{detect_risks(chunk)}`")
                entities = extract_ner_entities(chunk, ner)
                if entities:
                    st.markdown("**Entities Found:**")
                    st.json(entities)
                st.divider()


ImportError: cannot import name 'extract_text' from 'utils' (c:\Users\Anish\AppData\Local\Programs\Python\Python312\Lib\site-packages\utils\__init__.py)